# 07 • Transfert et ajustement fin hors ligne

`[MÉTA | Formation 4-024 | Niveau Application | TP 07 | Mode CPU local]`

**Objectif :** Vérifier concrètement le gel puis le dégel d’un extracteur.

**Temps indicatif :** 40 min. Ces temps sont répartis dans le conducteur, pas additionnés hors des 18 heures.

**Prérequis :** TP CNN.

**Preuves de réussite :** Source/cible disjointes, poids gelés inchangés, paramètres entraînables tracés.

**Sources :** R06 ; transposition pédagogique locale, pas modèle ImageNet.

Les jeux métier sont synthétiques. Aucun fichier personnel ou fiscal réel ne doit être chargé. Les résultats obtenus ici ne constituent pas une validation métier.

**Mode d’emploi :** exécuter les cellules dans l’ordre. Les cellules d’exercice du cahier apprenant sont à compléter ; le corrigé contient le code et des résultats de référence sur CPU.

In [ ]:
from pathlib import Path
import sys, os, json
# Chercher le kit depuis le répertoire du notebook ou celui de lancement.
HERE = Path.cwd().resolve()
TP_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "modules" / "atelier.py").exists()), None)
if TP_ROOT is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le dossier 03_Travaux_pratiques du kit décompressé.")
sys.path.insert(0, str(TP_ROOT / "modules"))
os.environ.setdefault("KERAS_BACKEND", "torch")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from atelier import *
seed_all(42)
print("Moteur disponible :", torch.__version__, "| Données :", DATA)


## 1. Deux domaines synthétiques distincts
Diviser le train en source propre et cible bruitée avant la transformation. La validation cible provient du split de validation original, jamais de la source. Le préentraînement est un petit modèle sur chiffres, pas une reproduction d’un grand réseau industriel.

In [ ]:
d=digits_data();X,y=d['train']
source_ix,cible_ix=train_test_split(np.arange(len(y)),test_size=.5,stratify=y,random_state=42)
assert not set(source_ix)&set(cible_ix)
rng=np.random.default_rng(72)
def bruit(images):return np.clip(images+rng.normal(0,.18,images.shape),0,1).astype(np.float32)
source=(X[source_ix],y[source_ix]);cible=(bruit(X[cible_ix]),y[cible_ix]);val=(bruit(d['validation'][0]),d['validation'][1])
seed_all();m=TinyCNN();h_source=fit_model(m,source,d['validation'],task='multi',epochs=12)
# Cette validation source sert au suivi et n'a provoqué ni early stopping ni choix hyperparamètre.
# L'évaluation cible finale indépendante restera sur le test réservé.
print('Source',len(source[1]),'cible',len(cible[1]),'validation cible',len(val[1]))

## 2. Nouvelle tête et extracteur gelé
Remplacer la tête, désactiver les gradients des caractéristiques, reconstruire l’optimiseur sur les seuls paramètres entraînables. Une comparaison des poids constitue une preuve plus solide qu’un cadenas dessiné sur une diapositive.

In [ ]:
# EXERCICE À COMPLÉTER
# Geler requires_grad avant fit_model. Vérifier que chaque poids de features reste identique.
# La correction est fournie séparément au formateur.
raise NotImplementedError("Compléter cette cellule puis relancer avant de poursuivre.")

## 3. Dégeler le dernier bloc
Le dernier Conv2d se trouve à l’indice 3 de features. Utiliser un taux plus faible. Cette architecture ne contient pas BatchNorm ; le problème de ses moyennes mobiles est traité dans TP 05, pas ignoré.

In [ ]:
# EXERCICE À COMPLÉTER
# Ne dégeler que la dernière convolution puis reconstruire l’optimiseur via un nouvel appel fit_model.
# La correction est fournie séparément au formateur.
raise NotImplementedError("Compléter cette cellule puis relancer avant de poursuivre.")

## 4. Interprétation et extensions
L’ajustement peut améliorer, stagner ou dégrader la validation. Décrire le résultat obtenu. Une comparaison depuis zéro constitue une extension, avec le même budget d’entraînement cible. L’usage des images de validation source pour suivre une courbe doit être distingué de l’ajustement sur leurs labels : aucun choix automatique n’est effectué ici. Pour un protocole scientifique complet, prévoir une validation source séparée également. Les grands poids préentraînés sont une autre variante à préparer institutionnellement, avec provenance, licence et téléchargement validés.